# Human Activity Recognition — ML Framework
**Author:** Meshari Saud Alaskar | [GitHub](https://github.com/Meshari-glitch)


In [ ]:
!pip install xgboost -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

print('✅ Libraries ready | TF:', tf.__version__)

## Load Dataset

In [ ]:
import urllib.request
import zipfile

print('Downloading dataset...')
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00240/UCI%20HAR%20Dataset.zip'
urllib.request.urlretrieve(url, 'har.zip')

with zipfile.ZipFile('har.zip', 'r') as z:
    z.extractall('.')

base = 'UCI HAR Dataset/'

X_train = pd.read_csv(base + 'train/X_train.txt', delim_whitespace=True, header=None)
X_test  = pd.read_csv(base + 'test/X_test.txt',  delim_whitespace=True, header=None)
y_train = pd.read_csv(base + 'train/y_train.txt', delim_whitespace=True, header=None, names=['Activity'])
y_test  = pd.read_csv(base + 'test/y_test.txt',  delim_whitespace=True, header=None, names=['Activity'])

activity_map = {1:'WALKING', 2:'WALKING_UPSTAIRS', 3:'WALKING_DOWNSTAIRS',
                4:'SITTING', 5:'STANDING', 6:'LAYING'}

y_train['Activity'] = y_train['Activity'].map(activity_map)
y_test['Activity']  = y_test['Activity'].map(activity_map)
y_train = y_train['Activity']
y_test  = y_test['Activity']

print(f'✅ Train: {X_train.shape} | Test: {X_test.shape}')
print(y_train.value_counts())

## EDA

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
y_train.value_counts().plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Train Distribution', fontweight='bold')
axes[0].tick_params(axis='x', rotation=30)
y_test.value_counts().plot(kind='bar', ax=axes[1], color='coral', edgecolor='black')
axes[1].set_title('Test Distribution', fontweight='bold')
axes[1].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

## Prepare Data

In [ ]:
le = LabelEncoder()
le.fit(y_train)
y_train_enc = le.transform(y_train)
y_test_enc  = le.transform(y_test)
y_train_cat = to_categorical(y_train_enc)
y_test_cat  = to_categorical(y_test_enc)
X_train_dl  = X_train.values.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test_dl   = X_test.values.reshape(X_test.shape[0],  X_test.shape[1],  1)
LABELS   = list(activity_map.values())
results  = {}
print('✅ Ready | DL shape:', X_train_dl.shape)

## Helper: Confusion Matrix

In [ ]:
def plot_cm(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred, labels=LABELS)
    fig, ax = plt.subplots(figsize=(8, 6))
    ConfusionMatrixDisplay(cm, display_labels=LABELS).plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(title, fontweight='bold')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()

## SVM

In [ ]:
print('Training SVM... (2-3 min)')
svm = SVC(kernel='rbf', C=10, gamma='scale', random_state=42)
svm.fit(X_train, y_train)
y_pred = svm.predict(X_test)
acc = accuracy_score(y_test, y_pred)
results['SVM'] = round(acc*100, 2)
print(f'✅ SVM: {acc*100:.2f}%')
print(classification_report(y_test, y_pred))
plot_cm(y_test, y_pred, 'Confusion Matrix — SVM')

## KNN

In [ ]:
print('Training KNN...')
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)
acc = accuracy_score(y_test, y_pred)
results['KNN'] = round(acc*100, 2)
print(f'✅ KNN: {acc*100:.2f}%')
print(classification_report(y_test, y_pred))
plot_cm(y_test, y_pred, 'Confusion Matrix — KNN')

## Decision Tree

In [ ]:
print('Training Decision Tree...')
dtc = DecisionTreeClassifier(max_depth=20, random_state=42)
dtc.fit(X_train, y_train)
y_pred = dtc.predict(X_test)
acc = accuracy_score(y_test, y_pred)
results['Decision Tree'] = round(acc*100, 2)
print(f'✅ Decision Tree: {acc*100:.2f}%')
print(classification_report(y_test, y_pred))
plot_cm(y_test, y_pred, 'Confusion Matrix — Decision Tree')

## Random Forest

In [ ]:
print('Training Random Forest...')
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
results['Random Forest'] = round(acc*100, 2)
print(f'✅ Random Forest: {acc*100:.2f}%')
print(classification_report(y_test, y_pred))
plot_cm(y_test, y_pred, 'Confusion Matrix — Random Forest')

## XGBoost

In [ ]:
print('Training XGBoost...')
xgb = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                    eval_metric='mlogloss', random_state=42, n_jobs=-1)
xgb.fit(X_train, y_train_enc)
y_pred = le.inverse_transform(xgb.predict(X_test))
acc = accuracy_score(y_test, y_pred)
results['XGBoost'] = round(acc*100, 2)
print(f'✅ XGBoost: {acc*100:.2f}%')
print(classification_report(y_test, y_pred))
plot_cm(y_test, y_pred, 'Confusion Matrix — XGBoost')

## CNN

In [ ]:
print('Training CNN...')
es = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
cnn = Sequential([
    Conv1D(64, 3, activation='relu', input_shape=(X_train_dl.shape[1], 1)),
    BatchNormalization(), MaxPooling1D(2),
    Conv1D(128, 3, activation='relu'),
    BatchNormalization(), MaxPooling1D(2),
    Flatten(), Dense(128, activation='relu'), Dropout(0.3),
    Dense(6, activation='softmax')
])
cnn.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
cnn.fit(X_train_dl, y_train_cat, epochs=15, batch_size=64,
        validation_data=(X_test_dl, y_test_cat), callbacks=[es], verbose=1)
y_pred = le.inverse_transform(np.argmax(cnn.predict(X_test_dl), axis=1))
acc = accuracy_score(y_test, y_pred)
results['CNN'] = round(acc*100, 2)
print(f'✅ CNN: {acc*100:.2f}%')
print(classification_report(y_test, y_pred))
plot_cm(y_test, y_pred, 'Confusion Matrix — CNN')

## LSTM

In [ ]:
print('Training LSTM...')
lstm = Sequential([
    LSTM(128, return_sequences=True, input_shape=(X_train_dl.shape[1], 1)),
    Dropout(0.3), LSTM(64), Dropout(0.3),
    Dense(64, activation='relu'),
    Dense(6, activation='softmax')
])
lstm.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
lstm.fit(X_train_dl, y_train_cat, epochs=15, batch_size=64,
         validation_data=(X_test_dl, y_test_cat), callbacks=[es], verbose=1)
y_pred = le.inverse_transform(np.argmax(lstm.predict(X_test_dl), axis=1))
acc = accuracy_score(y_test, y_pred)
results['LSTM'] = round(acc*100, 2)
print(f'✅ LSTM: {acc*100:.2f}%')
print(classification_report(y_test, y_pred))
plot_cm(y_test, y_pred, 'Confusion Matrix — LSTM')

## Final Comparison

In [ ]:
df = pd.DataFrame(list(results.items()), columns=['Model','Accuracy (%)']).sort_values('Accuracy (%)', ascending=False)
print('=' * 35)
print('     FINAL MODEL COMPARISON')
print('=' * 35)
print(df.to_string(index=False))
print('=' * 35)

colors = ['#2ecc71' if v == df['Accuracy (%)'].max() else '#3498db' for v in df['Accuracy (%)']]
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(df['Model'], df['Accuracy (%)'], color=colors, edgecolor='black', width=0.5)
for bar, val in zip(bars, df['Accuracy (%)']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
            f'{val}%', ha='center', fontweight='bold', fontsize=11)
ax.set_ylim([80, 100])
ax.set_title('Model Accuracy Comparison — HAR Project', fontsize=14, fontweight='bold')
ax.set_ylabel('Accuracy (%)')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()